# 02 · Databricks count watch
Scheduled job (03 creates it). Right count per pair on the Left as-of day; on a match: a ready-to-run dtrack config + email.

In [ ]:
dbutils.widgets.text("WATCH_ROOT", "/Volumes/<catalog>/<schema>/<volume>/count_watch")   # = S3_PREFIX of 01
dbutils.widgets.text("STATUS_TABLE", "<catalog>.<schema>.count_watch_status")
dbutils.widgets.text("LOOKBACK_DAYS", "7")
dbutils.widgets.dropdown("MODE", "watch", ["watch", "notify"])       # notify: the job's email task
dbutils.widgets.dropdown("NOTIFY_VIA", "job", ["job", "smtp"])       # job: email from the job notification
dbutils.widgets.text("SMTP_SCOPE", "count-watch")                    # smtp_host, smtp_port, smtp_user, smtp_password, mail_from, mail_to
dbutils.widgets.text("RUNS_ROOT", "s3://<bucket>/<prefix>/runs")     # for the commands in the email
W = {name: dbutils.widgets.get(name) for name in
     ["WATCH_ROOT", "STATUS_TABLE", "LOOKBACK_DAYS", "MODE", "NOTIFY_VIA", "SMTP_SCOPE", "RUNS_ROOT"]}
W

In [ ]:
import glob
import json
import os
from datetime import UTC, date, datetime, timedelta
from zoneinfo import ZoneInfo

config = json.load(open(f"{W['WATCH_ROOT']}/pairs.json"))
zone = ZoneInfo(config.get("timezone", "America/New_York"))
today = datetime.now(zone).date()
latest_path = f"{W['WATCH_ROOT']}/configs/latest.json"

if W["MODE"] == "notify":                         # the job emails on this task's success; show what is ready
    ready_config = json.load(open(latest_path))
    print(ready_config["_email"]["text"])
    dbutils.notebook.exit(json.dumps({"ready": list(ready_config["pairs"]), "config": ready_config["_email"]["path"]}))

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {W['STATUS_TABLE']} (
  pair STRING, as_of DATE, right_day DATE, left_count BIGINT, right_count BIGINT, state STRING,
  left_pulled_at STRING, checked_at TIMESTAMP, matched_at TIMESTAMP, notified_at TIMESTAMP, config_path STRING)
""")
notified = {(r.pair, r.as_of.isoformat()) for r in spark.table(W["STATUS_TABLE"]).where("notified_at IS NOT NULL").collect()}
since = (today - timedelta(days=int(W["LOOKBACK_DAYS"]))).isoformat()
snapshots = [json.load(open(path)) for path in sorted(glob.glob(f"{W['WATCH_ROOT']}/left/*/*.json"))]
pending = [s for s in snapshots if s["as_of"] >= since and s["pair"] in config["pairs"] and (s["pair"], s["as_of"]) not in notified]
print(f"{len(snapshots)} snapshot(s), {len(pending)} pending since {since}")

In [ ]:
def right_spec(pair):
    right = {"date_col": "dw_bus_dt", "date_type": "date", "offset_days": 0, **pair["right"]}
    return right


def right_literal(right, day_iso):
    kind = right["date_type"]
    if kind == "int":
        return day_iso.replace("-", "")
    return f"DATE '{day_iso}'" if kind == "date" else f"'{day_iso}'"


def right_count(pair, as_of):
    right = right_spec(pair)
    right_day = (date.fromisoformat(as_of) + timedelta(days=int(right["offset_days"]))).isoformat()
    filters = [f"`{right['date_col']}` = {right_literal(right, right_day)}"] + ([right["where"]] if right.get("where") else [])
    sql = f"SELECT COUNT(*) AS n FROM {right['table']} WHERE " + " AND ".join(f"({f})" for f in filters)
    return right_day, int(spark.sql(sql).collect()[0]["n"])


checked_at = datetime.now(UTC)
rows = []
for snap in pending:
    right_day, count = right_count(config["pairs"][snap["pair"]], snap["as_of"])
    state = "waiting_right" if count == 0 else ("match" if count == snap["left_count"] else "mismatch")
    rows.append((snap["pair"], date.fromisoformat(snap["as_of"]), date.fromisoformat(right_day), snap["left_count"],
                 count, state, snap["pulled_at"], checked_at, checked_at if state == "match" else None, None, None))
status = spark.createDataFrame(rows, spark.table(W["STATUS_TABLE"]).schema)
display(status.select("pair", "as_of", "right_day", "left_count", "right_count", "state"))

In [ ]:
# merge: keep the first matched_at; notified_at is set below, once per (pair, as_of)
status.createOrReplaceTempView("watch_now")
spark.sql(f"""
MERGE INTO {W['STATUS_TABLE']} t USING watch_now s ON t.pair = s.pair AND t.as_of = s.as_of
WHEN MATCHED THEN UPDATE SET right_day = s.right_day, left_count = s.left_count, right_count = s.right_count,
  state = s.state, left_pulled_at = s.left_pulled_at, checked_at = s.checked_at,
  matched_at = coalesce(t.matched_at, s.matched_at)
WHEN NOT MATCHED THEN INSERT *
""")
ready = [snap for snap, row in zip(pending, rows) if row[5] == "match"]
print("ready:", [(s["pair"], s["as_of"]) for s in ready])

In [ ]:
# a dtrack config per ready pair: the Left view gets a constant DTRACK_AS_OF column, the Right side is
# pinned to its day with `where`; date_synonyms aligns them when offset_days != 0; col_map by name
PASS_THROUGH = ("key", "key_families", "sample", "vintage", "strict", "col_type_overrides")


def dtrack_pair(snap):
    pair = config["pairs"][snap["pair"]]
    left, right = pair["left"], right_spec(pair)
    as_of = snap["as_of"]
    right_day = (date.fromisoformat(as_of) + timedelta(days=int(right["offset_days"]))).isoformat()
    base = left.get("query") or f"SELECT * FROM {left['table']}"
    filters = [left["where"]] if left.get("where") else []
    if snap["count_mode"] == "as_of":
        column = snap["as_of_rule"]["column"]
        on_day = f"TRUNC({column})" if left.get("source", "oracle") == "oracle" else f"CAST({column} AS DATE)"
        filters.append(f"{on_day} = DATE '{as_of}'")
    clause = (" WHERE " + " AND ".join(f"({f})" for f in filters)) if filters else ""
    dtrack_left = {
        "source": left.get("source", "oracle"),
        "query": f"SELECT t.*, DATE '{as_of}' AS DTRACK_AS_OF FROM ({base}) t{clause}",
        "date_col": "DTRACK_AS_OF",
        "date_type": "date",
        **({"conn_macro": left["conn"]} if left.get("conn") else {"path": left.get("path")}),
    }
    catalog, schema, table = right["table"].split(".")
    pinned = f"`{right['date_col']}` = {right_literal(right, right_day)}"
    dtrack_right = {
        "source": "databricks", "catalog": catalog, "schema": schema, "table": table,
        "date_col": right["date_col"], "date_type": {"date": "date", "int": "num"}.get(right["date_type"], "string_dash"),
        "where": " AND ".join(f"({w})" for w in [pinned, right.get("where")] if w),
    }
    right_columns = {c.lower(): c for c in spark.table(right["table"]).columns if c.lower() != right["date_col"].lower()}
    col_map = {k: v for k, v in (pair.get("col_map") or {}).items()}
    taken = {v.lower() for v in col_map.values()}
    for column in snap["columns"]:
        match = right_columns.get(column.lower())
        if column not in col_map and match and match.lower() not in taken:
            col_map[column] = match
            taken.add(match.lower())
    unmapped = {
        "left": [c for c in snap["columns"] if c not in col_map],
        "right": [c for c in right_columns.values() if c.lower() not in taken],
    }
    result = {"left": dtrack_left, "right": dtrack_right, "col_map": col_map}
    if right_day != as_of:
        result["date_synonyms"] = {right_day: as_of}
    result.update({k: pair[k] for k in PASS_THROUGH if k in pair})
    return result, unmapped


pairs, unmapped = {}, {}
for snap in ready:
    pairs[snap["pair"]], unmapped[snap["pair"]] = dtrack_pair(snap)
pairs = {name: spec for name, spec in pairs.items() if spec["col_map"]}
print(json.dumps(pairs, indent=2)[:4000])

In [ ]:
run_id = f"watch-{today.isoformat()}"
config_path = f"{W['WATCH_ROOT']}/configs/dtrack_{today.isoformat()}.json"
lines = [f"{len(pairs)} pair(s) ready for validation ({today}):"]
lines += [f"  {s['pair']}: as_of {s['as_of']}, rows {s['left_count']:,}" for s in ready if s["pair"] in pairs]
lines += ["", f"config: {config_path}", "",
          "validate today, before the Left view refreshes again:",
          f"  dtrack row-check --config dtrack_{today}.json --s3 {W['RUNS_ROOT']}/{run_id}",
          f"  (+ the Right job with run_id={run_id}), then: dtrack col-check --s3 {W['RUNS_ROOT']}/{run_id}",
          f"  or locally: python -m dtrack_local run --config dtrack_{today}.json --run-id {run_id}"]
lines += [f"  unmapped {name}: left {u['left']} | right {u['right']}" for name, u in unmapped.items() if u["left"] or u["right"]]
text = "\n".join(lines)
document = {"pairs": pairs}
if pairs:
    os.makedirs(os.path.dirname(config_path), exist_ok=True)
    with open(config_path, "w") as handle:
        json.dump(document, handle, indent=2)
    with open(latest_path, "w") as handle:
        json.dump({**document, "_email": {"text": text, "path": config_path}}, handle, indent=2)
print(text)

In [ ]:
def send_smtp(subject, body, attachment_name, attachment):
    import smtplib
    from email.message import EmailMessage

    secret = lambda key: dbutils.secrets.get(W["SMTP_SCOPE"], key)
    message = EmailMessage()
    message["Subject"], message["From"], message["To"] = subject, secret("mail_from"), secret("mail_to")
    message.set_content(body)
    message.add_attachment(attachment, maintype="application", subtype="json", filename=attachment_name)
    with smtplib.SMTP(secret("smtp_host"), int(secret("smtp_port"))) as server:
        server.starttls()
        server.login(secret("smtp_user"), secret("smtp_password"))
        server.send_message(message)


if pairs and W["NOTIFY_VIA"] == "smtp":
    send_smtp(f"[count watch] {len(pairs)} pair(s) ready {today}", text, f"dtrack_{today}.json",
              json.dumps(document, indent=2).encode())
if pairs:
    names = ", ".join(f"'{name}'" for name in pairs)
    spark.sql(f"""UPDATE {W['STATUS_TABLE']} SET notified_at = current_timestamp(), config_path = '{config_path}'
                  WHERE pair IN ({names}) AND state = 'match' AND notified_at IS NULL""")
dbutils.jobs.taskValues.set(key="ready", value=len(pairs))      # the job's condition task reads this
dbutils.notebook.exit(json.dumps({"ready": list(pairs), "checked": len(rows), "config": config_path if pairs else None}))